# LIME Explainability for BERT (WELFake)

This notebook applies **LIME** (Local Interpretable Model-agnostic Explanations)
to the best BERT model fine-tuned on the WELFake dataset.

**Workflow:**
1. Load saved BERT model from Google Drive
2. Reproduce the exact test split used during training
3. Run LIME on selected test samples
4. Export results as JSON for the local Streamlit viewer

In [1]:
!pip install lime -q

import pandas as pd
import torch
import numpy as np
import json
import os
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification
from lime.lime_text import LimeTextExplainer
from google.colab import drive, files

drive.mount('/content/drive')

Mounted at /content/drive


## Configuration

In [2]:
# ============================================================
# CONFIGURATION — Update these paths if needed
# ============================================================
MODEL_PATH = "/content/drive/MyDrive/bert_models/WELFake"
DATASET_PATH = "/content/drive/MyDrive/datasets/WELFake_processed.csv"

NUM_SAMPLES = 20          # Articles to explain (balanced: half real, half fake)
NUM_FEATURES = 20         # Top LIME features per explanation
NUM_PERTURBATIONS = 5000  # LIME perturbations per sample

# WELFake label encoding: 0 = Fake, 1 = Real
CLASS_NAMES = ['Fake', 'Real']

## Load Model & Data

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = BertForSequenceClassification.from_pretrained(MODEL_PATH)
tokenizer = BertTokenizer.from_pretrained(MODEL_PATH)
model.to(device)
model.eval()

print(f"✓ Model loaded from {MODEL_PATH}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

Using device: cuda


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✓ Model loaded from /content/drive/MyDrive/bert_models/WELFake
  Parameters: 109,483,778


In [4]:
df = pd.read_csv(DATASET_PATH).dropna()
print(f"✓ Dataset loaded: {len(df)} rows")
print(f"  Label distribution:\n{df['label'].value_counts()}")

# Reproduce the EXACT same 85/15 split from training (same random_state)
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['combined_text'].tolist(),
    df['label'].tolist(),
    test_size=0.15,
    random_state=42,
    stratify=df['label']
)
print(f"\n✓ Test set: {len(test_texts)} samples")
print(f"  Fake: {test_labels.count(0)}, Real: {test_labels.count(1)}")

✓ Dataset loaded: 63670 rows
  Label distribution:
label
0    34790
1    28880
Name: count, dtype: int64

✓ Test set: 9551 samples
  Fake: 5219, Real: 4332


## LIME Explainability

In [5]:
def bert_predict_proba(texts):
    """LIME-compatible prediction function.

    Takes a list of (potentially perturbed) text strings and returns
    an (N, 2) numpy array of probabilities: [P(Fake), P(Real)].
    """
    all_probs = []
    batch_size = 64

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        encodings = tokenizer(
            batch,
            truncation=True,
            padding=True,
            max_length=128,
            return_tensors='pt'
        )
        encodings = {k: v.to(device) for k, v in encodings.items()}

        with torch.no_grad():
            outputs = model(**encodings)
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
            all_probs.append(probs.cpu().numpy())

    return np.concatenate(all_probs, axis=0)

# Sanity check — verify probabilities are valid
test_probs = bert_predict_proba([test_texts[0]])
print(f"✓ Predictor sanity check:")
print(f"  P(Fake)={test_probs[0][0]:.4f}, P(Real)={test_probs[0][1]:.4f}")
print(f"  Sum={test_probs[0].sum():.4f} (should be 1.0000)")

✓ Predictor sanity check:
  P(Fake)=1.0000, P(Real)=0.0000
  Sum=1.0000 (should be 1.0000)


In [6]:
explainer = LimeTextExplainer(
    class_names=CLASS_NAMES,
    random_state=42
)

# Select a balanced set of test samples
real_indices = [i for i, l in enumerate(test_labels) if l == 1]
fake_indices = [i for i, l in enumerate(test_labels) if l == 0]

np.random.seed(42)
n_per_class = NUM_SAMPLES // 2
selected_real = np.random.choice(real_indices, size=n_per_class, replace=False)
selected_fake = np.random.choice(fake_indices, size=n_per_class, replace=False)
selected_indices = np.concatenate([selected_real, selected_fake])
np.random.shuffle(selected_indices)

print(f"Selected {len(selected_indices)} samples ({n_per_class} real, {n_per_class} fake)")
print()

explanations = []
for idx_num, idx in enumerate(selected_indices):
    text = test_texts[idx]
    true_label = test_labels[idx]
    label_name = CLASS_NAMES[true_label]

    print(f"[{idx_num+1}/{len(selected_indices)}] Sample {idx} — True: {label_name} ", end="", flush=True)

    exp = explainer.explain_instance(
        text,
        bert_predict_proba,
        num_features=NUM_FEATURES,
        num_samples=NUM_PERTURBATIONS,
        labels=(0, 1)
    )

    # Get model prediction for this sample
    probs = bert_predict_proba([text])[0]
    pred_label = int(np.argmax(probs))

    explanation_data = {
        "index": int(idx),
        "text": text[:5000],
        "true_label": int(true_label),
        "predicted_label": pred_label,
        "predicted_proba": probs.tolist(),
        "lime_weights_predicted": exp.as_list(label=pred_label),
        "lime_weights_class0": exp.as_list(label=0),
        "lime_weights_class1": exp.as_list(label=1),
    }
    explanations.append(explanation_data)

    status = "✓" if pred_label == true_label else "✗"
    print(f"→ Pred: {CLASS_NAMES[pred_label]} ({probs[pred_label]:.3f}) {status}")

correct = sum(1 for e in explanations if e['true_label'] == e['predicted_label'])
print(f"\n{'='*60}")
print(f"Done! {correct}/{len(explanations)} correct ({100*correct/len(explanations):.1f}%)")

Selected 20 samples (10 real, 10 fake)

[1/20] Sample 5229 — True: Fake → Pred: Fake (1.000) ✓
[2/20] Sample 8383 — True: Real → Pred: Real (1.000) ✓
[3/20] Sample 2262 — True: Real → Pred: Real (1.000) ✓
[4/20] Sample 5335 — True: Real → Pred: Real (1.000) ✓
[5/20] Sample 7339 — True: Fake → Pred: Fake (1.000) ✓
[6/20] Sample 5805 — True: Real → Pred: Real (1.000) ✓
[7/20] Sample 2590 — True: Real → Pred: Real (1.000) ✓
[8/20] Sample 6508 — True: Fake → Pred: Fake (0.821) ✓
[9/20] Sample 272 — True: Real → Pred: Real (1.000) ✓
[10/20] Sample 8223 — True: Real → Pred: Real (1.000) ✓
[11/20] Sample 8081 — True: Real → Pred: Real (1.000) ✓
[12/20] Sample 7150 — True: Fake → Pred: Fake (1.000) ✓
[13/20] Sample 6328 — True: Fake → Pred: Fake (1.000) ✓
[14/20] Sample 7837 — True: Fake → Pred: Fake (1.000) ✓
[15/20] Sample 1834 — True: Real → Pred: Real (1.000) ✓
[16/20] Sample 3131 — True: Fake → Pred: Fake (1.000) ✓
[17/20] Sample 5570 — True: Fake → Pred: Fake (1.000) ✓
[18/20] Sample 392

## Preview Results

In [7]:
# Preview first 5 explanations
for i, exp_data in enumerate(explanations[:5]):
    print(f"\n{'='*60}")
    tc = CLASS_NAMES[exp_data['true_label']]
    pc = CLASS_NAMES[exp_data['predicted_label']]
    conf = exp_data['predicted_proba'][exp_data['predicted_label']]
    status = "✓" if exp_data['true_label'] == exp_data['predicted_label'] else "✗ WRONG"

    print(f"Sample {i+1}: True={tc}, Pred={pc} ({conf:.3f}) {status}")
    print(f"Text: {exp_data['text'][:200]}...")
    print(f"\nTop 10 features (for predicted class '{pc}'):")
    for word, weight in exp_data['lime_weights_predicted'][:10]:
        bar = "█" * min(int(abs(weight) * 100), 40)
        sign = "+" if weight > 0 else "-"
        print(f"  {word:20s} {sign}{abs(weight):.4f} {bar}")


Sample 1: True=Fake, Pred=Fake (1.000) ✓
Text: U.S. Senate to hold hearing on Republican healthcare proposal WASHINGTON (Reuters) - A powerful U.S. Senate committee will hold a hearing on the latest proposed healthcare bill to overhaul Obamacare n...

Top 10 features (for predicted class 'Fake'):
  Reuters              +0.1177 ███████████
  WASHINGTON           +0.0728 ███████
  said                 +0.0511 █████
  S                    +0.0386 ███
  U                    +0.0379 ███
  Hatch                +0.0306 ███
  A                    +0.0292 ██
  Obamacare            +0.0264 ██
  overhaul             +0.0205 ██
  51                   +0.0190 █

Sample 2: True=Real, Pred=Real (1.000) ✓
Text: National Security Expert Warns Of The DIRE Dangers Of Trumps Plan For Intel Community (VIDEO) For the first time in our lives, we are entering an era where we have elected a president who is downright...

Top 10 features (for predicted class 'Real'):
  s                    +0.0000 
  VIDEO    

## Export Results

In [8]:
output = {
    "model_name": "BERT (bert-base-uncased)",
    "dataset_name": "WELFake",
    "model_path": MODEL_PATH,
    "num_samples": len(explanations),
    "num_features": NUM_FEATURES,
    "num_perturbations": NUM_PERTURBATIONS,
    "class_names": CLASS_NAMES,
    "explanations": explanations
}

output_path = "/content/lime_bert_welfake_results.json"
with open(output_path, 'w') as f:
    json.dump(output, f, indent=2)

file_size = os.path.getsize(output_path) / 1024
print(f"✓ Results saved to {output_path}")
print(f"  File size: {file_size:.1f} KB")
print(f"  Samples:   {len(explanations)}")

# Also save to Drive as backup
drive_output = "/content/drive/MyDrive/bert_models/lime_bert_welfake_results.json"
with open(drive_output, 'w') as f:
    json.dump(output, f, indent=2)
print(f"  Backup:    {drive_output}")

print(f"\nDownloading to your local machine...")
files.download(output_path)

✓ Results saved to /content/lime_bert_welfake_results.json
  File size: 145.7 KB
  Samples:   20
  Backup:    /content/drive/MyDrive/bert_models/lime_bert_welfake_results.json



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>